# STIR-Net V1 — 10 corrected same-sample overfit

This notebook validates the V1 corrective patch on the **same complete BlastoSPIM all-cell scene** used in the first overfit and debugging notebooks.

It starts from **fresh random initialization**. It does **not** load the old broken step-25 checkpoint.

The run is controlled:

1. verify the corrected source/configuration;
2. record a detailed step-0 baseline;
3. run a five-step safety gate;
4. if healthy, continue the same model to step 25;
5. track mask Dice, dense spatial geometry, structured matching, query existence, center trajectories, native support/prior behavior, and final instances;
6. compare against the original broken run.

A falling total loss alone is **not** success. The central question is whether actual cell-mask Dice starts improving.


In [ ]:
from pathlib import Path
import gc
import json
import subprocess
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

from learned.stirnet import RefinementCriterion, StirNet
from learned.stirnet.debugging.acceptance.first_overfit import (
    _reduced_config,
    _repo_root,
    build_real_batch,
)
from learned.stirnet.debugging.probes.matching import (
    coarse_dice_for_matches,
    run_matching_probe,
)
from learned.stirnet.model.native_masks import (
    native_chunk_coordinates_um,
    native_query_prior_and_support,
    source_dilation_support_chunk,
)
from learned.stirnet.model.query_builder import (
    QUERY_DISCOVERY,
    QUERY_PRIMARY,
    QUERY_SPLIT,
    QUERY_TEMPORAL,
)
from learned.stirnet.training.checkpoint import save_checkpoint
from learned.stirnet.training.trainer import move_to_device

SEED = 40266
TOTAL_STEPS = 25
EVAL_STEPS = {0, 1, 2, 3, 4, 5, 10, 15, 20, 25}
CHECKPOINT_STEPS = {5, 10, 15, 20, 25}
DEEP_STEPS = {0, 5, 25}

AMP_DTYPE = torch.float16
GRAD_SCALER_INITIAL_SCALE = 1024.0

PATCH_COMMIT = "2bc41f883d97fee2b3d8fe3d2a2c5cf6ca3f4748"

REPO_ROOT = _repo_root(Path.cwd())
DATA_DIR = (
    REPO_ROOT / "data" / "learned" / "stirnet" / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)
RUN_DIR = (
    REPO_ROOT / "runs" / "stirnet" / "first_overfit"
    / "10_corrected_same_sample"
)
RUN_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Notebook 10 requires CUDA.")

device = torch.device("cuda")
properties = torch.cuda.get_device_properties(0)

print("Repository     :", REPO_ROOT)
print("Data           :", DATA_DIR)
print("Run directory  :", RUN_DIR)
print("GPU            :", properties.name)
print("Dedicated VRAM :", f"{properties.total_memory / 1024**3:.3f} GiB")
print("PyTorch        :", torch.__version__)
print("CUDA runtime   :", torch.version.cuda)
print("Total steps    :", TOTAL_STEPS)


## 1. Source preflight

In [ ]:
def git_output(*args):
    try:
        result = subprocess.run(
            ["git", *args],
            cwd=REPO_ROOT,
            check=True,
            text=True,
            capture_output=True,
        )
        return result.stdout.strip()
    except Exception:
        return None

head = git_output("rev-parse", "HEAD")
print("Git HEAD:", head)

patch_present = None
try:
    result = subprocess.run(
        ["git", "merge-base", "--is-ancestor", PATCH_COMMIT, "HEAD"],
        cwd=REPO_ROOT,
        check=False,
    )
    patch_present = result.returncode == 0
except Exception:
    pass

print("Corrective patch ancestor of HEAD:", patch_present)

if patch_present is False:
    raise RuntimeError(
        "Expected corrective patch is not an ancestor of HEAD. "
        "Update the repository before running Notebook 10."
    )


## 2. Rebuild the exact all-cell BlastoSPIM scene

In [ ]:
batch, sample = build_real_batch(DATA_DIR)
cfg = _reduced_config()
target = batch["targets"][0]

print("Sample:", sample)
print("Spacing (um):", batch["spacing_um"][0].tolist())
print("dref (um):", float(batch["dref_um"][0]))

assert sample["current_count"] == 36
assert sample["target_count"] == 33
assert sample["temporal_tracklets"] == 52
assert sample["required_queries"] == 132

assert "source_ids" in target
assert "source_gt_overlap" in target

print("source_ids shape       :", tuple(target["source_ids"].shape))
print("source_gt_overlap shape:", tuple(target["source_gt_overlap"].shape))
print("positive compatibility :", int((target["source_gt_overlap"] > 0).sum()))


## 3. Verify the corrected configuration

In [ ]:
assert cfg.losses.overlap == 0.0
assert cfg.losses.mask_supervision_radius_dref == 1.5
assert cfg.losses.mask_focal_alpha_pos == 0.75
assert cfg.losses.mask_focal_gamma == 2.0

assert cfg.queries.native_support_radius_dref == 1.5
assert cfg.queries.native_source_dilation_dref == 0.5
assert cfg.queries.native_background_logit == -20.0
assert cfg.queries.prior_inside_logit == 1.5
assert cfg.queries.prior_outside_logit == -1.5

assert cfg.decoder.primary_center_step_dref == 0.50
assert cfg.decoder.split_center_step_dref == 0.75
assert cfg.decoder.temporal_center_step_dref == 0.25
assert cfg.decoder.discovery_center_step_dref == 1.00

print("Corrected config verified.")
print(json.dumps(cfg.to_dict(), indent=2))


## 4. Device batch and debug-capable forward

Targets remain CPU-backed. The spatial image is stored as FP16 on CUDA to preserve the RTX-4050 memory behavior.


In [ ]:
def prepare_device_batch(cpu_batch):
    result = {}
    for key, value in cpu_batch.items():
        if key == "targets":
            result[key] = value
        elif key == "spatial_inputs":
            result[key] = value.to(
                device=device,
                dtype=AMP_DTYPE,
                non_blocking=True,
            )
        elif key == "instance_labels":
            result[key] = value.to(
                device=device,
                dtype=torch.int32,
                non_blocking=True,
            )
        else:
            result[key] = move_to_device(value, device)
    return result


def model_forward_with_debug(model, b):
    return model(
        b["spatial_inputs"],
        b["instance_labels"],
        b["spacing_um"],
        b["dref_um"],
        b["instance_features"],
        b["instance_ids"],
        b["instance_batch"],
        b["instance_centroids_um"],
        b["graph_x"],
        b["graph_edge_index"],
        b["graph_edge_attr"],
        b["tracklet_id"],
        b["temporal_ref_um"],
        b["temporal_status"],
        b["hypothesis_edge_index"],
        b["hypothesis_edge_attr"],
        b["temporal_batch"],
        b.get("spatial_padding_mask"),
        return_debug=True,
    )


device_batch = prepare_device_batch(batch)

print("Spatial input :", tuple(device_batch["spatial_inputs"].shape), device_batch["spatial_inputs"].dtype)
print("Labels        :", tuple(device_batch["instance_labels"].shape), device_batch["instance_labels"].dtype)


## 5. Fresh model, criterion, optimizer

In [ ]:
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

model = StirNet(cfg).to(device)
criterion = RefinementCriterion(
    cfg.losses,
    cfg.queries,
    cfg.training,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.training.lr,
    weight_decay=cfg.training.weight_decay,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True,
    init_scale=GRAD_SCALER_INITIAL_SCALE,
)

print("Trainable parameters:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("Learning rate       :", cfg.training.lr)
print("Weight decay        :", cfg.training.weight_decay)
print("Gradient clip       :", cfg.training.max_grad_norm)
print("AMP scale           :", scaler.get_scale())


## 6. Query/matching diagnostics

In [ ]:
QUERY_NAMES = {
    QUERY_PRIMARY: "primary",
    QUERY_SPLIT: "split",
    QUERY_TEMPORAL: "temporal",
    QUERY_DISCOVERY: "discovery",
}


def source_overlap_count_for_query(outputs, q, target):
    source_id = int(outputs.source_instance_ids[0, q].detach().cpu())
    if source_id < 0:
        return 0

    source_ids = torch.as_tensor(target["source_ids"]).cpu().long()
    rows = torch.nonzero(source_ids == source_id, as_tuple=False).flatten()

    if not len(rows):
        return 0

    row = int(rows[0])

    return int(
        (
            torch.as_tensor(
                target["source_gt_overlap"]
            )[row] > 0
        ).sum()
    )


def matching_diagnostics(outputs, target):
    probe = run_matching_probe(outputs, [target])
    match = probe.matches[0]

    matched_set = set(
        match.pred_indices.detach().cpu().tolist()
    )

    query_types = (
        outputs.query_types[0]
        .detach()
        .cpu()
        .long()
    )

    source_ids_q = (
        outputs.source_instance_ids[0]
        .detach()
        .cpu()
        .long()
    )

    exist_prob = (
        torch.sigmoid(outputs.exist_logits[0])
        .detach()
        .float()
        .cpu()
    )

    valid = (
        ~outputs.query_padding_mask[0]
        .detach()
        .cpu()
    )

    target_source_ids = (
        torch.as_tensor(target["source_ids"])
        .cpu()
        .long()
    )

    overlap = (
        torch.as_tensor(
            target["source_gt_overlap"]
        )
        .cpu()
        .long()
    )

    source_row = {
        int(source_id): row
        for row, source_id
        in enumerate(target_source_ids.tolist())
    }

    incompatible_seeded = 0
    one_gt_split_positive = 0
    match_rows = []

    for q, t in zip(
        match.pred_indices.detach().cpu().tolist(),
        match.target_indices.detach().cpu().tolist(),
    ):
        q = int(q)
        t = int(t)

        qtype = int(query_types[q])
        sid = int(source_ids_q[q])

        compatible_count = 0
        compatible = None

        if sid >= 0 and sid in source_row:
            row = source_row[sid]
            compatible_count = int(
                (overlap[row] > 0).sum()
            )
            compatible = bool(
                overlap[row, t] > 0
            )

        if qtype in (
            QUERY_PRIMARY,
            QUERY_SPLIT,
        ):
            if compatible is not True:
                incompatible_seeded += 1

            if (
                qtype == QUERY_SPLIT
                and compatible_count == 1
            ):
                one_gt_split_positive += 1

        center_delta = (
            outputs.centers_cellscale[0, q]
            .detach()
            .float()
            .cpu()
            - torch.as_tensor(
                target["centers_cellscale"]
            )[t].float()
        )

        match_rows.append(
            {
                "query": q,
                "query_type": QUERY_NAMES[qtype],
                "target_index": t,
                "source_instance_id": sid,
                "source_overlap_gt_count": compatible_count,
                "source_compatible": compatible,
                "exist_prob": float(exist_prob[q]),
                "survives_0.5": bool(
                    exist_prob[q] > 0.5
                ),
                "center_error_um": float(
                    torch.linalg.vector_norm(
                        center_delta
                    )
                    * float(
                        outputs.dref_um[0]
                        .detach()
                        .cpu()
                    )
                ),
            }
        )

    existence_rows = []

    for qtype, name in QUERY_NAMES.items():
        ids = torch.nonzero(
            valid & (query_types == qtype),
            as_tuple=False,
        ).flatten()

        if not len(ids):
            continue

        is_matched = torch.tensor(
            [
                int(q) in matched_set
                for q in ids.tolist()
            ],
            dtype=torch.bool,
        )

        probs = exist_prob[ids]
        matched_probs = probs[is_matched]
        unmatched_probs = probs[~is_matched]

        existence_rows.append(
            {
                "query_type": name,
                "total": int(len(ids)),
                "matched": int(
                    is_matched.sum()
                ),
                "unmatched": int(
                    (~is_matched).sum()
                ),
                "surviving_total": int(
                    (probs > 0.5).sum()
                ),
                "matched_surviving": (
                    int(
                        (matched_probs > 0.5).sum()
                    )
                    if len(matched_probs)
                    else 0
                ),
                "unmatched_surviving": (
                    int(
                        (unmatched_probs > 0.5).sum()
                    )
                    if len(unmatched_probs)
                    else 0
                ),
                "matched_mean_exist": (
                    float(matched_probs.mean())
                    if len(matched_probs)
                    else np.nan
                ),
                "unmatched_mean_exist": (
                    float(unmatched_probs.mean())
                    if len(unmatched_probs)
                    else np.nan
                ),
            }
        )

    return {
        "probe": probe,
        "match_df": pd.DataFrame(
            match_rows
        ),
        "existence_df": pd.DataFrame(
            existence_rows
        ),
        "incompatible_seeded": (
            incompatible_seeded
        ),
        "one_gt_split_positive": (
            one_gt_split_positive
        ),
        "matched_count": int(
            len(match.pred_indices)
        ),
    }


## 7. Dense spatial metrics

In [ ]:
@torch.no_grad()
def binary_dense_metrics(
    logits,
    target_cpu,
    threshold=0.5,
    chunk_voxels=524_288,
):
    flat_logits = (
        logits.detach()
        .float()
        .reshape(-1)
    )
    flat_target = (
        torch.as_tensor(target_cpu)
        .reshape(-1)
    )

    intersection = 0
    pred_count = 0
    target_count = 0

    pos_sum = 0.0
    pos_count = 0

    neg_sum = 0.0
    neg_count = 0

    for start in range(
        0,
        flat_logits.numel(),
        chunk_voxels,
    ):
        end = min(
            start + chunk_voxels,
            flat_logits.numel(),
        )

        prob = torch.sigmoid(
            flat_logits[start:end]
        )

        target_chunk = (
            flat_target[start:end]
            .to(
                device=logits.device,
                dtype=torch.bool,
                non_blocking=True,
            )
        )

        pred = prob > threshold

        intersection += int(
            (pred & target_chunk)
            .sum()
            .cpu()
        )

        pred_count += int(
            pred.sum().cpu()
        )

        target_count += int(
            target_chunk.sum().cpu()
        )

        if target_chunk.any():
            pos_sum += float(
                prob[target_chunk]
                .sum()
                .cpu()
            )
            pos_count += int(
                target_chunk.sum().cpu()
            )

        negative = ~target_chunk

        if negative.any():
            neg_sum += float(
                prob[negative]
                .sum()
                .cpu()
            )
            neg_count += int(
                negative.sum().cpu()
            )

    return {
        "dice": (
            2.0 * intersection
            / max(
                pred_count + target_count,
                1,
            )
        ),
        "mean_prob_positive": (
            pos_sum
            / max(pos_count, 1)
        ),
        "mean_prob_negative": (
            neg_sum
            / max(neg_count, 1)
        ),
        "predicted_voxels": pred_count,
        "target_voxels": target_count,
    }


@torch.no_grad()
def dense_metrics(outputs, target):
    fg = binary_dense_metrics(
        outputs.dense_outputs[
            "foreground_logits"
        ][0, 0],
        target["foreground"],
    )

    boundary_target = (
        torch.as_tensor(
            target["boundary"]
        ) > 0.5
    )

    bd = binary_dense_metrics(
        outputs.dense_outputs[
            "boundary_logits"
        ][0, 0],
        boundary_target,
    )

    return {
        "foreground_dice": float(
            fg["dice"]
        ),
        "foreground_prob_inside": float(
            fg["mean_prob_positive"]
        ),
        "foreground_prob_outside": float(
            fg["mean_prob_negative"]
        ),
        "boundary_dice": float(
            bd["dice"]
        ),
        "boundary_prob_on": float(
            bd["mean_prob_positive"]
        ),
        "boundary_prob_off": float(
            bd["mean_prob_negative"]
        ),
    }


## 8. Center trajectory diagnostics

The current decoder exposes the initial query reference and all three refined references.

We record both the error to matched GT and the actual movement from the preceding layer.


In [ ]:
def center_trajectory_df(
    outputs,
    target,
    matching_probe,
):
    if outputs.debug is None:
        raise RuntimeError(
            "Debug references were not returned."
        )

    initial = (
        outputs.debug[
            "query_initial_references_cellscale"
        ][0]
        .detach()
        .float()
        .cpu()
    )

    layers = (
        outputs.debug[
            "query_layer_references_cellscale"
        ][:, 0]
        .detach()
        .float()
        .cpu()
    )

    dref = float(
        outputs.dref_um[0]
        .detach()
        .cpu()
    )

    qtypes = (
        outputs.query_types[0]
        .detach()
        .cpu()
        .long()
    )

    gt_centers = (
        torch.as_tensor(
            target["centers_cellscale"]
        )
        .float()
    )

    rows = []

    for q, t in zip(
        matching_probe.matches[0]
        .pred_indices
        .detach()
        .cpu()
        .tolist(),
        matching_probe.matches[0]
        .target_indices
        .detach()
        .cpu()
        .tolist(),
    ):
        q = int(q)
        t = int(t)

        refs = [
            initial[q],
            layers[0, q],
            layers[1, q],
            layers[2, q],
        ]

        gt = gt_centers[t]

        row = {
            "query": q,
            "query_type": QUERY_NAMES[
                int(qtypes[q])
            ],
            "target_index": t,
        }

        for index, ref in enumerate(refs):
            name = (
                "initial"
                if index == 0
                else f"layer{index}"
            )

            row[f"{name}_error_um"] = float(
                torch.linalg.vector_norm(
                    ref - gt
                )
                * dref
            )

        for index in range(
            1,
            len(refs),
        ):
            delta = (
                refs[index]
                - refs[index - 1]
            ) * dref

            row[
                f"layer{index}_step_norm_um"
            ] = float(
                torch.linalg.vector_norm(
                    delta
                )
            )

            row[
                f"layer{index}_step_max_axis_um"
            ] = float(
                delta.abs().max()
            )

        rows.append(row)

    return pd.DataFrame(rows)


## 9. Detailed evaluation snapshot

In [ ]:
@torch.no_grad()
def evaluate_snapshot(step):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model.eval()
    criterion.eval()

    started = time.perf_counter()

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        outputs = model_forward_with_debug(
            model,
            device_batch,
        )

        losses = criterion(
            outputs,
            device_batch["targets"],
        )

    torch.cuda.synchronize()

    loss_values = {
        key: float(
            value.detach()
            .float()
            .cpu()
        )
        for key, value
        in losses.items()
    }

    if not all(
        np.isfinite(value)
        for value in loss_values.values()
    ):
        raise RuntimeError(
            f"Non-finite evaluation loss "
            f"at step {step}: {loss_values}"
        )

    matching = matching_diagnostics(
        outputs,
        target,
    )

    if (
        matching[
            "incompatible_seeded"
        ] != 0
    ):
        raise RuntimeError(
            f"Incompatible seeded match "
            f"at step {step}."
        )

    if (
        matching[
            "one_gt_split_positive"
        ] != 0
    ):
        raise RuntimeError(
            f"One-GT split became positive "
            f"at step {step}."
        )

    if (
        matching["matched_count"]
        != sample["target_count"]
    ):
        raise RuntimeError(
            f"Expected "
            f"{sample['target_count']} "
            f"matched GT cells; got "
            f"{matching['matched_count']}."
        )

    if abs(
        loss_values["overlap"]
    ) > 1e-12:
        raise RuntimeError(
            "Overlap loss is not zero."
        )

    dense = dense_metrics(
        outputs,
        target,
    )

    match = (
        matching["probe"]
        .matches[0]
    )

    coarse_global = (
        coarse_dice_for_matches(
            outputs.coarse_mask_logits[0],
            target,
            match,
        )
    )

    center_df = center_trajectory_df(
        outputs,
        target,
        matching["probe"],
    )

    row = {
        "step": int(step),
        **loss_values,
        "supervised_coarse_dice": (
            1.0
            - loss_values[
                "dice_coarse"
            ]
        ),
        "supervised_native_dice": (
            1.0
            - loss_values[
                "dice_hi"
            ]
        ),
        "global_matched_coarse_dice_mean": (
            float(
                np.mean(
                    list(
                        coarse_global.values()
                    )
                )
            )
            if coarse_global
            else np.nan
        ),
        **dense,
        "matched_count": (
            matching[
                "matched_count"
            ]
        ),
        "incompatible_seeded": (
            matching[
                "incompatible_seeded"
            ]
        ),
        "one_gt_split_positive": (
            matching[
                "one_gt_split_positive"
            ]
        ),
        "seconds": float(
            time.perf_counter()
            - started
        ),
        "peak_cuda_gib": float(
            torch.cuda
            .max_memory_allocated()
            / 1024**3
        ),
    }

    print(
        f"eval step {step:03d} | "
        f"loss={row['loss']:.6f} | "
        f"coarse Dice="
        f"{row['supervised_coarse_dice']:.4f} | "
        f"native Dice="
        f"{row['supervised_native_dice']:.4f} | "
        f"fg={row['foreground_dice']:.4f} | "
        f"boundary="
        f"{row['boundary_dice']:.4f} | "
        f"peak="
        f"{row['peak_cuda_gib']:.2f} GiB"
    )

    return (
        row,
        matching,
        center_df,
        outputs,
    )


## 10. Native mask decomposition

For representative queries we measure:

- supported learned-only mask;
- supported prior-only mask;
- actual combined mask;
- fraction of matched GT voxels lying inside the true native render support.

This avoids misreading the old raw-prior diagnostic.


In [ ]:
class StreamMaskState:
    def __init__(self):
        self.intersection = 0.0
        self.probability_sum = 0.0
        self.target_sum = 0.0
        self.hard_intersection = 0
        self.hard_predicted = 0
        self.hard_target = 0

    def update(
        self,
        logits,
        target_bool,
        threshold=0.5,
    ):
        prob = torch.sigmoid(
            logits.float()
        )

        target_bool = (
            target_bool.bool()
        )

        self.intersection += float(
            (
                prob
                * target_bool.float()
            )
            .sum()
            .cpu()
        )

        self.probability_sum += float(
            prob.sum().cpu()
        )

        self.target_sum += float(
            target_bool.sum().cpu()
        )

        hard = prob > threshold

        self.hard_intersection += int(
            (hard & target_bool)
            .sum()
            .cpu()
        )

        self.hard_predicted += int(
            hard.sum().cpu()
        )

        self.hard_target += int(
            target_bool.sum().cpu()
        )

    def summary(
        self,
        prefix,
    ):
        soft = (
            2.0 * self.intersection
            + 1e-6
        ) / (
            self.probability_sum
            + self.target_sum
            + 1e-6
        )

        hard = (
            2.0
            * self.hard_intersection
            + 1e-6
        ) / (
            self.hard_predicted
            + self.hard_target
            + 1e-6
        )

        return {
            f"{prefix}_soft_dice": float(
                soft
            ),
            f"{prefix}_hard_dice": float(
                hard
            ),
            f"{prefix}_predicted_voxels": int(
                self.hard_predicted
            ),
            f"{prefix}_target_voxels": int(
                self.hard_target
            ),
            f"{prefix}_volume_ratio": (
                self.hard_predicted
                / max(
                    self.hard_target,
                    1,
                )
            ),
        }


def select_representative_queries(
    outputs,
    target,
    matching,
):
    match = (
        matching["probe"]
        .matches[0]
    )

    matched_map = {
        int(q): int(t)
        for q, t in zip(
            match.pred_indices
            .detach()
            .cpu()
            .tolist(),
            match.target_indices
            .detach()
            .cpu()
            .tolist(),
        )
    }

    qtypes = (
        outputs.query_types[0]
        .detach()
        .cpu()
        .long()
    )

    valid = (
        ~outputs.query_padding_mask[0]
        .detach()
        .cpu()
    )

    probs = (
        torch.sigmoid(
            outputs.exist_logits[0]
        )
        .detach()
        .float()
        .cpu()
    )

    categories = {}

    def choose(
        name,
        candidates,
    ):
        candidates = list(
            candidates
        )

        if not candidates:
            return

        categories[name] = max(
            candidates,
            key=lambda q: float(
                probs[q]
            ),
        )

    choose(
        "matched_primary_one_gt",
        [
            q
            for q in matched_map
            if (
                int(qtypes[q])
                == QUERY_PRIMARY
                and source_overlap_count_for_query(
                    outputs,
                    q,
                    target,
                )
                == 1
            )
        ],
    )

    choose(
        "matched_split_merge",
        [
            q
            for q in matched_map
            if (
                int(qtypes[q])
                == QUERY_SPLIT
                and source_overlap_count_for_query(
                    outputs,
                    q,
                    target,
                )
                >= 2
            )
        ],
    )

    choose(
        "matched_temporal",
        [
            q
            for q in matched_map
            if int(qtypes[q])
            == QUERY_TEMPORAL
        ],
    )

    choose(
        "matched_discovery",
        [
            q
            for q in matched_map
            if int(qtypes[q])
            == QUERY_DISCOVERY
        ],
    )

    choose(
        "unmatched_split",
        [
            int(q)
            for q in torch.nonzero(
                valid
                & (
                    qtypes
                    == QUERY_SPLIT
                ),
                as_tuple=False,
            ).flatten().tolist()
            if int(q)
            not in matched_map
        ],
    )

    choose(
        "unmatched_discovery",
        [
            int(q)
            for q in torch.nonzero(
                valid
                & (
                    qtypes
                    == QUERY_DISCOVERY
                ),
                as_tuple=False,
            ).flatten().tolist()
            if int(q)
            not in matched_map
        ],
    )

    return (
        categories,
        matched_map,
    )


@torch.no_grad()
def native_query_decomposition(
    outputs,
    target,
    q,
    target_index,
    chunk_voxels=None,
):
    if chunk_voxels is None:
        chunk_voxels = int(
            cfg.losses.native_chunk_voxels
        )

    b = 0

    (
        _,
        channels,
        z_size,
        y_size,
        x_size,
    ) = outputs.mask_features.shape

    feature_flat = (
        outputs.mask_features[b]
        .reshape(
            channels,
            -1,
        )
    )

    voxel_count = (
        feature_flat.shape[1]
    )

    embedding = (
        outputs.native_mask_embeddings[
            b,
            q,
        ]
    )

    qtype = (
        outputs.query_types[
            b,
            q : q + 1,
        ]
    )

    source_id = (
        outputs.source_instance_ids[
            b,
            q : q + 1,
        ]
    )

    ref_um = (
        outputs.centers_cellscale[
            b,
            q : q + 1,
        ].float()
        * outputs.dref_um[b].float()
    )

    learned_state = StreamMaskState()
    prior_state = StreamMaskState()
    combined_state = StreamMaskState()

    support_target_voxels = 0
    target_voxels = 0

    gt_id = None

    if target_index is not None:
        gt_id = int(
            torch.as_tensor(
                target["ids"]
            )[target_index]
        )

    for start in range(
        0,
        voxel_count,
        chunk_voxels,
    ):
        end = min(
            start + chunk_voxels,
            voxel_count,
        )

        learned_logits = torch.einsum(
            "c,cv->v",
            embedding,
            feature_flat[
                :,
                start:end,
            ],
        ).float()

        coords_um = (
            native_chunk_coordinates_um(
                (
                    z_size,
                    y_size,
                    x_size,
                ),
                outputs.spacing_um[b],
                start,
                end,
            )
        )

        source_support = (
            source_dilation_support_chunk(
                outputs.instance_labels[b],
                source_id,
                outputs.spacing_um[b],
                outputs.dref_um[b],
                cfg.queries
                .native_source_dilation_dref,
                start,
                end,
            )
        )

        current_chunk = (
            outputs.instance_labels[b]
            .reshape(-1)[start:end]
        )

        prior, support = (
            native_query_prior_and_support(
                qtype,
                source_id,
                ref_um,
                current_chunk,
                source_support,
                coords_um,
                outputs.dref_um[b],
                support_radius_dref=(
                    cfg.queries
                    .native_support_radius_dref
                ),
                temporal_sigma_dref=(
                    cfg.queries
                    .temporal_gaussian_sigma_dref
                ),
                prior_inside_logit=(
                    cfg.queries
                    .prior_inside_logit
                ),
                prior_outside_logit=(
                    cfg.queries
                    .prior_outside_logit
                ),
            )
        )

        support = support[0]
        prior = prior[0]

        background = float(
            cfg.queries
            .native_background_logit
        )

        supported_learned = (
            learned_logits.masked_fill(
                ~support,
                background,
            )
        )

        supported_prior = (
            prior.masked_fill(
                ~support,
                background,
            )
        )

        combined = (
            learned_logits + prior
        ).masked_fill(
            ~support,
            background,
        )

        if gt_id is None:
            target_chunk = (
                torch.zeros(
                    end - start,
                    device=(
                        learned_logits.device
                    ),
                    dtype=torch.bool,
                )
            )
        else:
            labels_cpu = (
                torch.as_tensor(
                    target["label_map"]
                )
                .reshape(-1)[
                    start:end
                ]
            )

            target_chunk = (
                labels_cpu.to(
                    device=(
                        learned_logits.device
                    ),
                    non_blocking=True,
                )
                == gt_id
            )

        learned_state.update(
            supported_learned,
            target_chunk,
        )

        prior_state.update(
            supported_prior,
            target_chunk,
        )

        combined_state.update(
            combined,
            target_chunk,
        )

        if gt_id is not None:
            target_voxels += int(
                target_chunk
                .sum()
                .cpu()
            )

            support_target_voxels += int(
                (
                    support
                    & target_chunk
                )
                .sum()
                .cpu()
            )

    row = {
        "query": int(q),
        "query_type": QUERY_NAMES[
            int(qtype.item())
        ],
        "target_index": (
            int(target_index)
            if target_index
            is not None
            else -1
        ),
        "exist_prob": float(
            torch.sigmoid(
                outputs.exist_logits[
                    b,
                    q,
                ]
            )
            .detach()
            .float()
            .cpu()
        ),
        "support_gt_coverage": (
            (
                support_target_voxels
                / max(
                    target_voxels,
                    1,
                )
            )
            if gt_id is not None
            else np.nan
        ),
    }

    row.update(
        learned_state.summary(
            "supported_learned"
        )
    )

    row.update(
        prior_state.summary(
            "supported_prior"
        )
    )

    row.update(
        combined_state.summary(
            "combined"
        )
    )

    return row


@torch.no_grad()
def deep_native_snapshot(
    step,
    outputs,
    matching,
):
    (
        categories,
        matched_map,
    ) = (
        select_representative_queries(
            outputs,
            target,
            matching,
        )
    )

    rows = []

    for category, q in (
        categories.items()
    ):
        row = (
            native_query_decomposition(
                outputs,
                target,
                q,
                matched_map.get(q),
            )
        )

        row["step"] = int(step)
        row["category"] = category
        rows.append(row)

    df = pd.DataFrame(rows)

    print(
        f"\nRepresentative native masks "
        f"at step {step}:"
    )

    if len(df):
        display(
            df[
                [
                    "category",
                    "query",
                    "query_type",
                    "target_index",
                    "exist_prob",
                    "support_gt_coverage",
                    "supported_learned_hard_dice",
                    "supported_learned_volume_ratio",
                    "supported_prior_hard_dice",
                    "supported_prior_volume_ratio",
                    "combined_hard_dice",
                    "combined_volume_ratio",
                ]
            ]
        )
    else:
        print(
            "No representative queries found."
        )

    return df


## 11. Training-step helper

In [ ]:
def gradient_coverage(model):
    total = 0
    finite = 0
    nonzero = 0

    for parameter in (
        model.parameters()
    ):
        if parameter.grad is None:
            continue

        total += 1
        grad = parameter.grad.detach()

        if torch.isfinite(
            grad
        ).all():
            finite += 1

        if (
            torch.count_nonzero(
                grad
            ).item()
            > 0
        ):
            nonzero += 1

    return (
        total,
        finite,
        nonzero,
    )


def train_one_step(step):
    model.train()
    criterion.train()

    optimizer.zero_grad(
        set_to_none=True
    )

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    started = time.perf_counter()

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        outputs = (
            model_forward_with_debug(
                model,
                device_batch,
            )
        )

        losses = criterion(
            outputs,
            device_batch["targets"],
        )

        loss = losses["loss"]

    if not torch.isfinite(
        loss.detach().float()
    ):
        raise RuntimeError(
            f"Non-finite training loss "
            f"at step {step}."
        )

    scaler.scale(
        loss
    ).backward()

    scaler.unscale_(
        optimizer
    )

    (
        total_grads,
        finite_grads,
        nonzero_grads,
    ) = gradient_coverage(model)

    if (
        finite_grads
        != total_grads
    ):
        raise RuntimeError(
            f"Non-finite gradients "
            f"at step {step}: "
            f"{finite_grads}/"
            f"{total_grads} finite"
        )

    preclip_norm = (
        torch.nn.utils
        .clip_grad_norm_(
            model.parameters(),
            cfg.training
            .max_grad_norm,
        )
    )

    scaler.step(
        optimizer
    )

    scaler.update()

    torch.cuda.synchronize()

    row = {
        "step": int(step),
        **{
            key: float(
                value.detach()
                .float()
                .cpu()
            )
            for key, value
            in losses.items()
        },
        "grad_tensors": int(
            total_grads
        ),
        "finite_grad_tensors": int(
            finite_grads
        ),
        "nonzero_grad_tensors": int(
            nonzero_grads
        ),
        "preclip_grad_norm": float(
            torch.as_tensor(
                preclip_norm
            )
            .detach()
            .float()
            .cpu()
        ),
        "amp_scale": float(
            scaler.get_scale()
        ),
        "seconds": float(
            time.perf_counter()
            - started
        ),
        "peak_cuda_gib": float(
            torch.cuda
            .max_memory_allocated()
            / 1024**3
        ),
    }

    del (
        outputs,
        losses,
        loss,
    )

    gc.collect()
    torch.cuda.empty_cache()

    return row


## 12. Step-0 baseline

In [ ]:
eval_history = []
train_history = []
existence_history = []
center_history = []
deep_history = []

(
    row0,
    matching0,
    center0,
    outputs0,
) = evaluate_snapshot(0)

eval_history.append(row0)

exist0 = (
    matching0[
        "existence_df"
    ].copy()
)

exist0["step"] = 0
existence_history.append(
    exist0
)

center0 = center0.copy()
center0["step"] = 0
center_history.append(
    center0
)

deep0 = deep_native_snapshot(
    0,
    outputs0,
    matching0,
)

deep_history.append(
    deep0
)

print(
    "\nStep-0 existence calibration:"
)
display(
    matching0[
        "existence_df"
    ]
)

print(
    "\nStep-0 matched composition:"
)

display(
    matching0[
        "match_df"
    ]
    .groupby(
        "query_type"
    )
    .agg(
        matched=(
            "query",
            "count",
        ),
        surviving=(
            "survives_0.5",
            "sum",
        ),
        mean_exist=(
            "exist_prob",
            "mean",
        ),
        mean_center_error_um=(
            "center_error_um",
            "mean",
        ),
    )
    .reset_index()
)

del outputs0

gc.collect()
torch.cuda.empty_cache()


## 13. Five-step safety gate, then continue the same model to step 25

Hard-stop conditions are narrow:

- non-finite losses/gradients;
- incompatible seeded matching;
- positive split from a one-GT source;
- failure to match all 33 GT cells;
- nonzero overlap loss.

The notebook does not stop merely because Dice is still modest at step 5.


In [ ]:
for step in range(
    1,
    TOTAL_STEPS + 1,
):
    train_row = (
        train_one_step(
            step
        )
    )

    train_history.append(
        train_row
    )

    print(
        f"train step {step:03d} | "
        f"loss="
        f"{train_row['loss']:.6f} | "
        f"grad="
        f"{train_row['preclip_grad_norm']:.4f} | "
        f"finite="
        f"{train_row['finite_grad_tensors']}/"
        f"{train_row['grad_tensors']} | "
        f"nonzero="
        f"{train_row['nonzero_grad_tensors']} | "
        f"{train_row['seconds']:.1f}s"
    )

    if step in EVAL_STEPS:
        (
            eval_row,
            matching,
            center_step_df,
            outputs_eval,
        ) = evaluate_snapshot(
            step
        )

        eval_history.append(
            eval_row
        )

        exist_step_df = (
            matching[
                "existence_df"
            ].copy()
        )

        exist_step_df[
            "step"
        ] = step

        existence_history.append(
            exist_step_df
        )

        center_step_df = (
            center_step_df.copy()
        )

        center_step_df[
            "step"
        ] = step

        center_history.append(
            center_step_df
        )

        if step in DEEP_STEPS:
            deep_step_df = (
                deep_native_snapshot(
                    step,
                    outputs_eval,
                    matching,
                )
            )

            deep_history.append(
                deep_step_df
            )

        if step == 5:
            print(
                "\nFIVE-STEP SAFETY GATE PASSED."
            )
            print(
                "Continuing the same fresh model "
                "to step 25."
            )

        del outputs_eval

        gc.collect()
        torch.cuda.empty_cache()

    if step in CHECKPOINT_STEPS:
        checkpoint_path = (
            RUN_DIR
            / f"checkpoint_step_{step:03d}.pt"
        )

        save_checkpoint(
            checkpoint_path,
            model=model,
            optimizer=optimizer,
            scaler=scaler,
            step=step,
            epoch=0,
            config=cfg,
            extra={
                "experiment": (
                    "corrected same-sample "
                    "overfit"
                ),
                "patch_commit": (
                    PATCH_COMMIT
                ),
                "seed": SEED,
            },
        )

        print(
            "Saved checkpoint:",
            checkpoint_path,
        )


## 14. Consolidate and save histories

In [ ]:
eval_df = pd.DataFrame(
    eval_history
)

train_df = pd.DataFrame(
    train_history
)

existence_df = pd.concat(
    existence_history,
    ignore_index=True,
)

center_df = pd.concat(
    center_history,
    ignore_index=True,
)

deep_frames = [
    frame
    for frame in deep_history
    if (
        frame is not None
        and len(frame)
    )
]

deep_df = (
    pd.concat(
        deep_frames,
        ignore_index=True,
    )
    if deep_frames
    else pd.DataFrame()
)

eval_df.to_csv(
    RUN_DIR
    / "evaluation_history.csv",
    index=False,
)

train_df.to_csv(
    RUN_DIR
    / "training_history.csv",
    index=False,
)

existence_df.to_csv(
    RUN_DIR
    / "existence_history.csv",
    index=False,
)

center_df.to_csv(
    RUN_DIR
    / "center_trajectory_history.csv",
    index=False,
)

if len(deep_df):
    deep_df.to_csv(
        RUN_DIR
        / "native_decomposition_history.csv",
        index=False,
    )

with (
    RUN_DIR / "history.json"
).open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        {
            "evaluation": eval_history,
            "training": train_history,
        },
        handle,
        indent=2,
    )

print(
    "Saved histories to:",
    RUN_DIR,
)

display(eval_df)


## 15. Core learning curves

The important quantity is not only total loss:

```text
supervised Dice = 1 - Dice loss
```

The old broken run reduced total loss while actual Dice stayed nearly zero.


In [ ]:
plt.figure(
    figsize=(8, 5)
)

plt.plot(
    eval_df["step"],
    eval_df["loss"],
    marker="o",
)

plt.xlabel(
    "Optimizer step"
)
plt.ylabel(
    "Total evaluation loss"
)
plt.title(
    "Corrected same-sample total loss"
)
plt.grid(alpha=0.2)
plt.show()


plt.figure(
    figsize=(8, 5)
)

plt.plot(
    eval_df["step"],
    eval_df[
        "supervised_coarse_dice"
    ],
    marker="o",
    label=(
        "coarse supervised Dice"
    ),
)

plt.plot(
    eval_df["step"],
    eval_df[
        "supervised_native_dice"
    ],
    marker="o",
    label=(
        "native supervised Dice"
    ),
)

plt.xlabel(
    "Optimizer step"
)
plt.ylabel("Dice")
plt.title(
    "Instance-mask learning"
)
plt.legend()
plt.grid(alpha=0.2)
plt.show()


plt.figure(
    figsize=(8, 5)
)

plt.plot(
    eval_df["step"],
    eval_df[
        "foreground_dice"
    ],
    marker="o",
    label="foreground Dice",
)

plt.plot(
    eval_df["step"],
    eval_df[
        "boundary_dice"
    ],
    marker="o",
    label="boundary Dice",
)

plt.xlabel(
    "Optimizer step"
)
plt.ylabel("Dice")
plt.title(
    "Dense spatial geometry"
)
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 16. Query existence calibration over training

In [ ]:
display(
    existence_df.sort_values(
        [
            "step",
            "query_type",
        ]
    )
)

plt.figure(
    figsize=(9, 5)
)

for query_type in (
    QUERY_NAMES.values()
):
    rows = existence_df[
        existence_df[
            "query_type"
        ]
        == query_type
    ]

    if not len(rows):
        continue

    plt.plot(
        rows["step"],
        rows[
            "unmatched_mean_exist"
        ],
        marker="o",
        label=(
            f"{query_type} unmatched"
        ),
    )

plt.axhline(
    0.5,
    linestyle="--",
    linewidth=1,
)

plt.xlabel(
    "Optimizer step"
)
plt.ylabel(
    "Mean existence probability"
)
plt.title(
    "Unmatched-query existence calibration"
)
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 17. Center-refinement behavior

Pay particular attention to temporal queries whose initial Trackastra-derived reference starts within 1 µm of the matched GT center.


In [ ]:
summary_rows = []

for (
    step,
    query_type,
), group in center_df.groupby(
    [
        "step",
        "query_type",
    ]
):
    accurate_initial = group[
        group[
            "initial_error_um"
        ] < 1.0
    ]

    summary_rows.append(
        {
            "step": int(step),
            "query_type": query_type,
            "count": len(group),
            "mean_initial_error_um": (
                group[
                    "initial_error_um"
                ].mean()
            ),
            "mean_final_error_um": (
                group[
                    "layer3_error_um"
                ].mean()
            ),
            "mean_L1_step_norm_um": (
                group[
                    "layer1_step_norm_um"
                ].mean()
            ),
            "mean_L2_step_norm_um": (
                group[
                    "layer2_step_norm_um"
                ].mean()
            ),
            "mean_L3_step_norm_um": (
                group[
                    "layer3_step_norm_um"
                ].mean()
            ),
            "accurate_initial_count": (
                len(accurate_initial)
            ),
            "accurate_initial_mean_final_error_um": (
                accurate_initial[
                    "layer3_error_um"
                ].mean()
                if len(
                    accurate_initial
                )
                else np.nan
            ),
        }
    )

center_summary_df = (
    pd.DataFrame(
        summary_rows
    )
)

display(
    center_summary_df.sort_values(
        [
            "step",
            "query_type",
        ]
    )
)

center_summary_df.to_csv(
    RUN_DIR
    / "center_summary.csv",
    index=False,
)


## 18. Native decomposition through training

Main columns:

- `support_gt_coverage`
- `supported_learned_hard_dice`
- `combined_hard_dice`
- corresponding volume ratios.


In [ ]:
if len(deep_df):
    display(
        deep_df[
            [
                "step",
                "category",
                "query",
                "query_type",
                "exist_prob",
                "support_gt_coverage",
                "supported_learned_soft_dice",
                "supported_learned_hard_dice",
                "supported_learned_volume_ratio",
                "supported_prior_hard_dice",
                "supported_prior_volume_ratio",
                "combined_soft_dice",
                "combined_hard_dice",
                "combined_volume_ratio",
            ]
        ]
        .sort_values(
            [
                "category",
                "step",
            ]
        )
    )
else:
    print(
        "No deep native rows produced."
    )


## 19. Direct comparison with the old broken run

These are fixed diagnostic reference values from the original step-25 debugging session. The old checkpoint is never loaded.


In [ ]:
OLD_REFERENCE = {
    "step0_total_loss": 35.2161,
    "step25_total_loss": 9.451205,
    "step25_dice_coarse_loss": 0.9990468,
    "step25_dice_hi_loss": 0.9973469,
    "step25_overlap": 0.0026913,
    "old_full_foreground_dice": 0.0528651,
    "old_full_boundary_dice": 0.0494871,
}

final_eval = (
    eval_df[
        eval_df["step"]
        == TOTAL_STEPS
    ]
    .iloc[0]
)

comparison = pd.DataFrame(
    [
        {
            "metric": (
                "coarse supervised Dice"
            ),
            "old_step25": (
                1.0
                - OLD_REFERENCE[
                    "step25_dice_coarse_loss"
                ]
            ),
            "corrected_step25": (
                final_eval[
                    "supervised_coarse_dice"
                ]
            ),
        },
        {
            "metric": (
                "native supervised Dice"
            ),
            "old_step25": (
                1.0
                - OLD_REFERENCE[
                    "step25_dice_hi_loss"
                ]
            ),
            "corrected_step25": (
                final_eval[
                    "supervised_native_dice"
                ]
            ),
        },
        {
            "metric": (
                "dense foreground Dice"
            ),
            "old_step25": (
                OLD_REFERENCE[
                    "old_full_foreground_dice"
                ]
            ),
            "corrected_step25": (
                final_eval[
                    "foreground_dice"
                ]
            ),
        },
        {
            "metric": (
                "dense boundary Dice"
            ),
            "old_step25": (
                OLD_REFERENCE[
                    "old_full_boundary_dice"
                ]
            ),
            "corrected_step25": (
                final_eval[
                    "boundary_dice"
                ]
            ),
        },
        {
            "metric": "overlap loss",
            "old_step25": (
                OLD_REFERENCE[
                    "step25_overlap"
                ]
            ),
            "corrected_step25": (
                final_eval[
                    "overlap"
                ]
            ),
        },
    ]
)

display(comparison)

comparison.to_csv(
    RUN_DIR
    / "old_vs_corrected.csv",
    index=False,
)


## 20. Memory-safe final inference

The repository postprocessor can render all selected native masks together. On a 6 GB laptop GPU that is unnecessarily risky for this diagnostic.

This notebook reproduces the same final semantics **one query at a time**:

- final existence threshold;
- native rendering;
- mask threshold;
- connected component nearest predicted center;
- per-voxel winner by `existence × mask probability`.

This avoids a simultaneous `[num_selected, Z, Y, X]` tensor.


In [ ]:
def component_near_center(
    mask,
    center_vox,
    min_voxels,
):
    cc, count = ndi.label(mask)

    if count == 0:
        return np.zeros_like(
            mask,
            dtype=bool,
        )

    center = np.rint(
        center_vox
    ).astype(int)

    if (
        np.all(center >= 0)
        and np.all(
            center
            < np.asarray(
                mask.shape
            )
        )
    ):
        label = int(
            cc[tuple(center)]
        )

        if (
            label > 0
            and np.count_nonzero(
                cc == label
            )
            >= min_voxels
        ):
            return cc == label

    objects = ndi.find_objects(
        cc
    )

    best_label = None
    best_distance = np.inf

    for label, slices in enumerate(
        objects,
        start=1,
    ):
        if slices is None:
            continue

        component = (
            cc[slices]
            == label
        )

        if (
            int(
                component.sum()
            )
            < min_voxels
        ):
            continue

        local_coords = (
            np.argwhere(
                component
            )
        )

        offset = np.asarray(
            [
                s.start
                for s in slices
            ],
            dtype=float,
        )

        coords = (
            local_coords
            + offset[None]
        )

        distance = float(
            np.linalg.norm(
                coords
                - center[None],
                axis=1,
            ).min()
        )

        if (
            distance
            < best_distance
        ):
            best_distance = (
                distance
            )
            best_label = label

    return (
        cc == best_label
        if best_label is not None
        else np.zeros_like(
            mask,
            dtype=bool,
        )
    )


@torch.no_grad()
def streamed_postprocess_single_query(
    model,
    outputs,
):
    probs = (
        torch.sigmoid(
            outputs.exist_logits[0]
        )
        .masked_fill(
            outputs.query_padding_mask[0],
            0,
        )
        .detach()
        .float()
        .cpu()
    )

    selected = torch.nonzero(
        probs
        > cfg.inference
        .final_exist_threshold,
        as_tuple=False,
    ).flatten()

    shape = tuple(
        int(v)
        for v
        in outputs
        .instance_labels
        .shape[-3:]
    )

    spacing = (
        outputs.spacing_um[0]
        .detach()
        .float()
        .cpu()
        .numpy()
    )

    dref = float(
        outputs.dref_um[0]
        .detach()
        .cpu()
    )

    extent = (
        np.asarray(
            shape,
            dtype=float,
        )
        - 1
    ) * spacing

    labels = np.zeros(
        shape,
        dtype=np.int32,
    )

    best_score = np.full(
        shape,
        -np.inf,
        dtype=np.float32,
    )

    accepted = []

    for q in selected.tolist():
        index = torch.tensor(
            [q],
            device=(
                outputs
                .exist_logits
                .device
            ),
            dtype=torch.long,
        )

        rendered = (
            model.render_masks(
                outputs,
                [index],
            )[0][0]
        )

        probability = (
            torch.sigmoid(
                rendered.float()
            )
            .cpu()
            .numpy()
            .astype(
                np.float32,
                copy=False,
            )
        )

        mask = (
            probability
            > cfg.inference
            .mask_threshold
        )

        center_rel_um = (
            outputs
            .centers_cellscale[
                0,
                q,
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            * dref
        )

        center_vox = (
            center_rel_um
            + 0.5 * extent
        ) / spacing

        mask = (
            component_near_center(
                mask,
                center_vox,
                cfg.inference
                .min_mask_voxels,
            )
        )

        if (
            int(mask.sum())
            < cfg.inference
            .min_mask_voxels
        ):
            del (
                rendered,
                probability,
                mask,
            )

            torch.cuda.empty_cache()
            continue

        score = (
            float(probs[q])
            * probability
        )

        update = (
            mask
            & (
                score
                > best_score
            )
        )

        label_id = (
            len(accepted)
            + 1
        )

        labels[update] = (
            label_id
        )

        best_score[update] = (
            score[update]
        )

        accepted.append(
            {
                "query": int(q),
                "query_type": (
                    QUERY_NAMES[
                        int(
                            outputs
                            .query_types[
                                0,
                                q,
                            ]
                            .detach()
                            .cpu()
                        )
                    ]
                ),
                "exist_prob": float(
                    probs[q]
                ),
                "mask_voxels_before_conflict": int(
                    mask.sum()
                ),
            }
        )

        del (
            rendered,
            probability,
            mask,
            score,
            update,
        )

        gc.collect()
        torch.cuda.empty_cache()

    return (
        labels,
        pd.DataFrame(
            accepted
        ),
    )


## 21. Final fresh forward + streamed postprocessing

In [ ]:
model.eval()
criterion.eval()

gc.collect()
torch.cuda.empty_cache()

with torch.no_grad(), torch.autocast(
    device_type="cuda",
    dtype=AMP_DTYPE,
):
    final_outputs = (
        model_forward_with_debug(
            model,
            device_batch,
        )
    )

(
    pred_labels,
    accepted_df,
) = streamed_postprocess_single_query(
    model,
    final_outputs,
)

gt_labels = (
    torch.as_tensor(
        target["label_map"]
    )
    .cpu()
    .numpy()
    .astype(
        np.int32,
        copy=False,
    )
)

gt_fg = gt_labels > 0
pred_fg = pred_labels > 0

foreground_dice = (
    2.0
    * np.count_nonzero(
        gt_fg & pred_fg
    )
    / max(
        np.count_nonzero(
            gt_fg
        )
        + np.count_nonzero(
            pred_fg
        ),
        1,
    )
)

print(
    "Final predicted instances:",
    int(pred_labels.max()),
)

print(
    "GT instances             :",
    len(
        np.unique(
            gt_labels[
                gt_labels > 0
            ]
        )
    ),
)

print(
    "Final foreground Dice    :",
    foreground_dice,
)

print(
    "\nAccepted query types:"
)

if len(accepted_df):
    display(
        accepted_df.groupby(
            "query_type"
        )
        .agg(
            accepted=(
                "query",
                "count",
            ),
            mean_exist=(
                "exist_prob",
                "mean",
            ),
        )
        .reset_index()
    )
else:
    print(
        "No accepted queries."
    )


## 22. GT↔prediction instance Dice

This is an evaluation-only Hungarian assignment between final predicted labels and GT labels. It is unrelated to the model's training matcher.


In [ ]:
def instance_dice_evaluation(
    pred_labels,
    gt_labels,
):
    (
        pred_ids,
        pred_counts,
    ) = np.unique(
        pred_labels[
            pred_labels > 0
        ],
        return_counts=True,
    )

    (
        gt_ids,
        gt_counts,
    ) = np.unique(
        gt_labels[
            gt_labels > 0
        ],
        return_counts=True,
    )

    pred_row = {
        int(label): row
        for row, label
        in enumerate(
            pred_ids.tolist()
        )
    }

    gt_col = {
        int(label): col
        for col, label
        in enumerate(
            gt_ids.tolist()
        )
    }

    intersections = np.zeros(
        (
            len(pred_ids),
            len(gt_ids),
        ),
        dtype=np.int64,
    )

    positive = (
        (pred_labels > 0)
        & (gt_labels > 0)
    )

    if positive.any():
        (
            pairs,
            counts,
        ) = np.unique(
            np.stack(
                [
                    pred_labels[
                        positive
                    ],
                    gt_labels[
                        positive
                    ],
                ],
                axis=1,
            ),
            axis=0,
            return_counts=True,
        )

        for (
            pred_id,
            gt_id,
        ), count in zip(
            pairs.tolist(),
            counts.tolist(),
        ):
            intersections[
                pred_row[
                    int(pred_id)
                ],
                gt_col[
                    int(gt_id)
                ],
            ] = int(count)

    dice = (
        2.0
        * intersections
        / np.maximum(
            pred_counts[
                :,
                None,
            ]
            + gt_counts[
                None,
                :,
            ],
            1,
        )
    )

    if dice.size:
        rows, cols = (
            linear_sum_assignment(
                1.0 - dice
            )
        )

        values = dice[
            rows,
            cols,
        ]
    else:
        rows = np.empty(
            0,
            dtype=int,
        )

        cols = np.empty(
            0,
            dtype=int,
        )

        values = np.empty(
            0,
            dtype=float,
        )

    table = pd.DataFrame(
        [
            {
                "pred_id": int(
                    pred_ids[row]
                ),
                "gt_id": int(
                    gt_ids[col]
                ),
                "dice": float(
                    value
                ),
            }
            for (
                row,
                col,
                value,
            ) in zip(
                rows,
                cols,
                values,
            )
        ]
    )

    summary = {
        "pred_count": int(
            len(pred_ids)
        ),
        "gt_count": int(
            len(gt_ids)
        ),
        "hungarian_pairs": int(
            len(values)
        ),
        "positive_dice_pairs": int(
            np.count_nonzero(
                values > 0
            )
        ),
        "mean_matched_dice": (
            float(
                values.mean()
            )
            if len(values)
            else 0.0
        ),
        "median_matched_dice": (
            float(
                np.median(
                    values
                )
            )
            if len(values)
            else 0.0
        ),
        "mean_positive_matched_dice": (
            float(
                values[
                    values > 0
                ].mean()
            )
            if np.any(
                values > 0
            )
            else 0.0
        ),
    }

    return (
        summary,
        table,
    )


(
    instance_summary,
    instance_pairs_df,
) = instance_dice_evaluation(
    pred_labels,
    gt_labels,
)

print(instance_summary)

display(
    instance_pairs_df
    .sort_values(
        "dice",
        ascending=False,
    )
)


## 23. Visual sanity check

In [ ]:
raw = (
    batch["spatial_inputs"][
        0,
        0,
    ]
    .float()
    .cpu()
    .numpy()
)

z = raw.shape[0] // 2

fig, axes = plt.subplots(
    1,
    3,
    figsize=(16, 5),
)

axes[0].imshow(
    raw[z],
    cmap="gray",
)

axes[0].set_title(
    f"Raw z={z}"
)

axes[1].imshow(
    gt_labels[z]
)

axes[1].set_title(
    "GT instances"
)

axes[2].imshow(
    pred_labels[z]
)

axes[2].set_title(
    "Corrected STIR-Net prediction"
)

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


## 24. Optional Napari 3D view

In [ ]:
OPEN_NAPARI = True

if OPEN_NAPARI:
    import napari

    spacing_zyx = tuple(
        float(v)
        for v
        in batch[
            "spacing_um"
        ][0]
    )

    viewer = napari.Viewer(
        ndisplay=3
    )

    viewer.add_image(
        raw,
        name="Raw",
        scale=spacing_zyx,
    )

    viewer.add_labels(
        gt_labels,
        name="GT",
        scale=spacing_zyx,
    )

    viewer.add_labels(
        pred_labels,
        name=(
            "STIR-Net corrected"
        ),
        scale=spacing_zyx,
    )

    viewer.add_labels(
        batch["instance_labels"][0].cpu().numpy().astype(np.int32),
        name="Current CC input",
        scale=spacing_zyx,
    )

    viewer.add_image(
        (gt_labels > 0).astype(np.uint8),
        name="GT foreground",
        scale=spacing_zyx,
        visible=False,
    )

    napari.run()
else:
    print(
        "Napari skipped."
    )


## 25. Final diagnostic decision table

These are simple diagnostic gates, not production acceptance criteria. The trajectories and visual result above remain the main evidence.


In [ ]:
step0 = (
    eval_df[
        eval_df["step"]
        == 0
    ]
    .iloc[0]
)

step25 = (
    eval_df[
        eval_df["step"]
        == TOTAL_STEPS
    ]
    .iloc[0]
)

final_exist = (
    existence_df[
        existence_df["step"]
        == TOTAL_STEPS
    ]
)

discovery_row = (
    final_exist[
        final_exist[
            "query_type"
        ]
        == "discovery"
    ]
)

unmatched_discovery_survivors = (
    int(
        discovery_row.iloc[0][
            "unmatched_surviving"
        ]
    )
    if len(discovery_row)
    else 0
)

support_ok = True

if len(deep_df):
    matched_deep = deep_df[
        (
            deep_df["step"]
            == TOTAL_STEPS
        )
        & (
            deep_df[
                "target_index"
            ]
            >= 0
        )
    ]

    if len(matched_deep):
        support_ok = bool(
            (
                matched_deep[
                    "support_gt_coverage"
                ]
                >= 0.95
            ).all()
        )

decision = pd.DataFrame(
    [
        {
            "gate": (
                "source-incompatible "
                "seeded matches"
            ),
            "value": int(
                step25[
                    "incompatible_seeded"
                ]
            ),
            "desired": "0",
            "pass": (
                int(
                    step25[
                        "incompatible_seeded"
                    ]
                )
                == 0
            ),
        },
        {
            "gate": (
                "one-GT positive "
                "split matches"
            ),
            "value": int(
                step25[
                    "one_gt_split_positive"
                ]
            ),
            "desired": "0",
            "pass": (
                int(
                    step25[
                        "one_gt_split_positive"
                    ]
                )
                == 0
            ),
        },
        {
            "gate": "overlap loss",
            "value": float(
                step25["overlap"]
            ),
            "desired": "0",
            "pass": (
                abs(
                    float(
                        step25[
                            "overlap"
                        ]
                    )
                )
                < 1e-12
            ),
        },
        {
            "gate": (
                "coarse supervised "
                "Dice change"
            ),
            "value": float(
                step25[
                    "supervised_coarse_dice"
                ]
                - step0[
                    "supervised_coarse_dice"
                ]
            ),
            "desired": "> 0",
            "pass": (
                step25[
                    "supervised_coarse_dice"
                ]
                > step0[
                    "supervised_coarse_dice"
                ]
            ),
        },
        {
            "gate": (
                "native supervised "
                "Dice change"
            ),
            "value": float(
                step25[
                    "supervised_native_dice"
                ]
                - step0[
                    "supervised_native_dice"
                ]
            ),
            "desired": "> 0",
            "pass": (
                step25[
                    "supervised_native_dice"
                ]
                > step0[
                    "supervised_native_dice"
                ]
            ),
        },
        {
            "gate": (
                "dense foreground "
                "not catastrophic"
            ),
            "value": float(
                step25[
                    "foreground_dice"
                ]
            ),
            "desired": (
                "> 0.20 diagnostic floor"
            ),
            "pass": (
                float(
                    step25[
                        "foreground_dice"
                    ]
                )
                > 0.20
            ),
        },
        {
            "gate": (
                "representative "
                "render-support coverage"
            ),
            "value": support_ok,
            "desired": (
                ">= 95% for matched "
                "representatives"
            ),
            "pass": support_ok,
        },
        {
            "gate": (
                "unmatched discovery "
                "survivors"
            ),
            "value": (
                unmatched_discovery_survivors
            ),
            "desired": (
                "< 8 old failure"
            ),
            "pass": (
                unmatched_discovery_survivors
                < 8
            ),
        },
    ]
)

display(decision)

decision.to_csv(
    RUN_DIR
    / "diagnostic_decision.csv",
    index=False,
)


## 26. Persist final inference summary and release CUDA

In [ ]:
accepted_df.to_csv(
    RUN_DIR
    / "accepted_queries.csv",
    index=False,
)

instance_pairs_df.to_csv(
    RUN_DIR
    / "final_instance_dice_pairs.csv",
    index=False,
)

np.save(
    RUN_DIR
    / "final_pred_labels.npy",
    pred_labels.astype(
        np.int32,
        copy=False,
    ),
)

final_summary = {
    "patch_commit": (
        PATCH_COMMIT
    ),
    "git_head": head,
    "seed": SEED,
    "sample": sample,
    "final_eval": {
        key: (
            float(value)
            if isinstance(
                value,
                (
                    np.floating,
                    float,
                ),
            )
            else int(value)
            if isinstance(
                value,
                (
                    np.integer,
                    int,
                ),
            )
            else value
        )
        for key, value
        in step25.to_dict().items()
    },
    "final_foreground_dice": float(
        foreground_dice
    ),
    "final_instance_evaluation": (
        instance_summary
    ),
    "accepted_query_count": int(
        len(accepted_df)
    ),
}

with (
    RUN_DIR
    / "final_summary.json"
).open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        final_summary,
        handle,
        indent=2,
    )

print(
    "Final artifacts saved under:"
)
print(RUN_DIR)

del final_outputs

model.cpu()

del (
    model,
    criterion,
    optimizer,
    scaler,
    device_batch,
)

gc.collect()
torch.cuda.empty_cache()

print(
    "CUDA cache cleared."
)


# Interpretation

### Corrective patch succeeded

The strongest pattern is:

```text
coarse/native supervised Dice rises materially
dense foreground/boundary remain healthy
primary queries are no longer universally suppressed
legitimate split queries survive only for merged sources
unmatched discovery queries no longer all survive
combined native masks have sane volume ratios
good temporal references remain reasonably stable
final instance prediction becomes biologically recognizable
```

If this happens, the next engineering step is the **permanent staged training curriculum**, not another architecture redesign.

### Mask Dice improves but final rendering is poor

Inspect native support coverage, prior/support calibration, and connected-component postprocessing before changing CNN/GNN/CR.

### Dense spatial geometry collapses again

Then the corrected query objective still conflicts with the shared backbone. The next intervention is staged optimization and/or differential learning rates.

### Query masks remain poor while dense geometry is healthy

Then focus on the query mask representation/decoder and native support semantics.

### Matching stays correct but existence remains wrong

Then isolate existence/count calibration rather than changing the spatial or temporal architecture.


In [ ]:
# ============================================================
# DEBUG VISUALIZATION:
#   1. All Hungarian-matched masks, IGNORING existence threshold
#   2. All queries with existence > 0.30
# ============================================================

import gc
import numpy as np
import torch
import napari

from learned.stirnet import StirNet
from learned.stirnet.training.checkpoint import load_checkpoint
from learned.stirnet.debugging.probes.matching import run_matching_probe


# ------------------------------------------------------------
# 1. Recreate the step-25 model/output if Notebook 10 cleanup
#    has already deleted them.
# ------------------------------------------------------------

checkpoint_path = RUN_DIR / "checkpoint_step_025.pt"

print("Loading:", checkpoint_path)

model_debug = StirNet(cfg).to(device)

load_checkpoint(
    checkpoint_path,
    model_debug,
    map_location=device,
    strict=True,
)

model_debug.eval()

device_batch_debug = prepare_device_batch(batch)

gc.collect()
torch.cuda.empty_cache()

with torch.no_grad(), torch.autocast(
    device_type="cuda",
    dtype=AMP_DTYPE,
):
    outputs_debug = model_forward_with_debug(
        model_debug,
        device_batch_debug,
    )

print("Step-25 output reconstructed.")


# ------------------------------------------------------------
# 2. Get the final structured Hungarian matching.
# ------------------------------------------------------------

matching_debug = run_matching_probe(
    outputs_debug,
    [target],
)

match = matching_debug.matches[0]

matched_queries = (
    match.pred_indices
    .detach()
    .cpu()
    .long()
)

matched_targets = (
    match.target_indices
    .detach()
    .cpu()
    .long()
)

exist_probs = (
    torch.sigmoid(outputs_debug.exist_logits[0])
    .detach()
    .float()
    .cpu()
)

query_types_cpu = (
    outputs_debug.query_types[0]
    .detach()
    .cpu()
    .long()
)

valid_queries = (
    ~outputs_debug.query_padding_mask[0]
    .detach()
    .cpu()
)

loose_queries = torch.nonzero(
    valid_queries & (exist_probs > 0.30),
    as_tuple=False,
).flatten()

print("Hungarian-matched queries :", len(matched_queries))
print("Queries with exist > 0.30 :", len(loose_queries))
print("Queries with exist > 0.50 :", int((valid_queries & (exist_probs > 0.50)).sum()))


# ------------------------------------------------------------
# 3. Memory-safe renderer.
#
# For the matched-mask layer:
#   - existence is NOT used for filtering
#   - existence is NOT used in voxel competition
#
# This lets us inspect the mask branch separately from the
# existence branch.
# ------------------------------------------------------------

@torch.no_grad()
def render_debug_query_set(
    model,
    outputs,
    query_indices,
    *,
    use_existence_in_score=False,
):
    shape = tuple(
        int(v)
        for v in outputs.instance_labels.shape[-3:]
    )

    spacing = (
        outputs.spacing_um[0]
        .detach()
        .float()
        .cpu()
        .numpy()
    )

    dref_um = float(
        outputs.dref_um[0]
        .detach()
        .cpu()
    )

    extent_um = (
        np.asarray(shape, dtype=np.float32) - 1
    ) * spacing

    labels = np.zeros(
        shape,
        dtype=np.int32,
    )

    best_score = np.full(
        shape,
        -np.inf,
        dtype=np.float32,
    )

    accepted_rows = []

    for output_label, q in enumerate(
        query_indices.tolist(),
        start=1,
    ):
        q = int(q)

        selected = torch.tensor(
            [q],
            device=outputs.exist_logits.device,
            dtype=torch.long,
        )

        # Render only one native mask at a time.
        rendered = model.render_masks(
            outputs,
            [selected],
        )[0][0]

        probability = (
            torch.sigmoid(rendered.float())
            .detach()
            .cpu()
            .numpy()
            .astype(np.float32, copy=False)
        )

        binary = (
            probability
            > cfg.inference.mask_threshold
        )

        center_rel_um = (
            outputs.centers_cellscale[0, q]
            .detach()
            .float()
            .cpu()
            .numpy()
            * dref_um
        )

        center_vox = (
            center_rel_um
            + 0.5 * extent_um
        ) / spacing

        # Same connected-component cleanup used in Notebook 10.
        binary = component_near_center(
            binary,
            center_vox,
            cfg.inference.min_mask_voxels,
        )

        mask_voxels = int(binary.sum())

        if mask_voxels < cfg.inference.min_mask_voxels:
            accepted_rows.append(
                {
                    "query": q,
                    "query_type": QUERY_NAMES[int(query_types_cpu[q])],
                    "exist_prob": float(exist_probs[q]),
                    "mask_voxels": mask_voxels,
                    "rendered": False,
                }
            )

            del rendered, probability, binary
            torch.cuda.empty_cache()
            continue

        if use_existence_in_score:
            score = (
                float(exist_probs[q])
                * probability
            )
        else:
            # Important:
            # inspect mask quality independently of existence.
            score = probability

        update = (
            binary
            & (score > best_score)
        )

        labels[update] = output_label
        best_score[update] = score[update]

        accepted_rows.append(
            {
                "query": q,
                "query_type": QUERY_NAMES[int(query_types_cpu[q])],
                "exist_prob": float(exist_probs[q]),
                "mask_voxels": mask_voxels,
                "rendered": True,
            }
        )

        del rendered, probability, binary, score, update

        gc.collect()
        torch.cuda.empty_cache()

    return labels, accepted_rows


# ------------------------------------------------------------
# 4. Render ALL 33 matched queries with existence ignored.
# ------------------------------------------------------------

print("\nRendering all matched queries, ignoring existence...")

matched_mask_labels, matched_mask_rows = render_debug_query_set(
    model_debug,
    outputs_debug,
    matched_queries,
    use_existence_in_score=False,
)

print(
    "Non-empty matched masks:",
    sum(row["rendered"] for row in matched_mask_rows),
    "/",
    len(matched_mask_rows),
)


# ------------------------------------------------------------
# 5. Render all queries above the loose 0.30 existence threshold.
# ------------------------------------------------------------

print("\nRendering queries with existence > 0.30...")

loose_mask_labels, loose_mask_rows = render_debug_query_set(
    model_debug,
    outputs_debug,
    loose_queries,
    use_existence_in_score=True,
)

print(
    "Non-empty loose masks:",
    sum(row["rendered"] for row in loose_mask_rows),
    "/",
    len(loose_mask_rows),
)


# ------------------------------------------------------------
# 6. Print exactly which matched queries were suppressed by
#    the normal 0.50 existence threshold.
# ------------------------------------------------------------

matched_table = []

for q, target_index in zip(
    matched_queries.tolist(),
    matched_targets.tolist(),
):
    q = int(q)

    matched_table.append(
        {
            "query": q,
            "type": QUERY_NAMES[int(query_types_cpu[q])],
            "target_index": int(target_index),
            "exist_prob": float(exist_probs[q]),
            "passes_0.30": bool(exist_probs[q] > 0.30),
            "passes_0.50": bool(exist_probs[q] > 0.50),
        }
    )

matched_table_df = pd.DataFrame(matched_table)

display(
    matched_table_df.sort_values(
        "exist_prob",
        ascending=False,
    )
)

print(
    "\nMatched queries rejected only because exist <= 0.50:",
    int((~matched_table_df["passes_0.50"]).sum()),
    "/",
    len(matched_table_df),
)


# ------------------------------------------------------------
# 7. Add the diagnostic layers to the existing Napari viewer.
# ------------------------------------------------------------

spacing_zyx = tuple(
    float(v)
    for v in batch["spacing_um"][0]
)

# Reuse the currently open viewer if possible.
try:
    viewer
except NameError:
    viewer = napari.Viewer(ndisplay=3)

    viewer.add_image(
        raw,
        name="Raw",
        scale=spacing_zyx,
    )

    viewer.add_labels(
        gt_labels,
        name="GT",
        scale=spacing_zyx,
    )


# Remove old versions if this cell is rerun.
for layer_name in [
    "Matched masks - ignore existence",
    "Queries exist > 0.30",
]:
    if layer_name in viewer.layers:
        viewer.layers.remove(layer_name)


viewer.add_labels(
    matched_mask_labels,
    name="Matched masks - ignore existence",
    scale=spacing_zyx,
    opacity=0.75,
)

viewer.add_labels(
    loose_mask_labels,
    name="Queries exist > 0.30",
    scale=spacing_zyx,
    opacity=0.75,
    visible=False,
)

print("\nAdded Napari layers:")
print("  Matched masks - ignore existence")
print("  Queries exist > 0.30")